In [12]:
import pandas as pd

df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [13]:
# Tokenize

def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [14]:
tokenize("Hello my name is vamshi")

['hello', 'my', 'name', 'is', 'vamshi']

In [15]:
vocab = {'<UNK>':0} # Total unique words  #<unk>->unknown

In [16]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenzied_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenzied_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [17]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [18]:
len(vocab)

324

In [ ]:
vocab

In [21]:
#Convert words to numerical indices

def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text


In [22]:
text_to_indices("my name is vamshi", vocab)

[0, 0, 2, 0]

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

In [26]:
class QADataSet(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]


  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [27]:
dataset = QADataSet(df, vocab)

In [28]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [41]:
torch.manual_seed(42)
for question, answer in dataloader:
  print(f"question: {question}, answer: {answer[0]}")

question: tensor([[  1,   2,   3,  17, 115,  83,  84]]), answer: tensor([116])
question: tensor([[ 10, 308,   3, 309, 310]]), answer: tensor([311])
question: tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]), answer: tensor([244])
question: tensor([[ 1,  2,  3,  4,  5, 73]]), answer: tensor([74])
question: tensor([[ 10,  96,   3, 104, 239]]), answer: tensor([240])
question: tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]), answer: tensor([155])
question: tensor([[78, 79, 80, 81, 82, 83, 84]]), answer: tensor([85])
question: tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]), answer: tensor([199])
question: tensor([[10, 96,  3, 97]]), answer: tensor([98])
question: tensor([[ 42, 137,   2, 138,  39, 175, 269]]), answer: tensor([99])
question: tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]), answer: tensor([28])
question: tensor([[  1,   2,   3,  33,  34,   5, 245]]), answer: tensor([246])
question: tensor([[  1,   2,   3,  37,  38,  39, 161]]), answer: tensor([162])
question: tens

In [45]:
len(dataloader)

90

In [42]:
import torch.nn as nn

In [47]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(input_size=50, hidden_size=64, batch_first=True) #Input same as embedding_dim ig
    self.fc = nn.Linear(in_features=64, out_features=vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))


    return output


In [50]:
!pip install torchinfo
from torchinfo import summary

In [64]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("Shape of a: ", a.shape)

b = x(a)
print("Shape of b: ", b.shape)

c,d = y(b)
print("shape of c:", c.shape)
print("shape of d: ", d.shape)

e = z(d.squeeze(0))

print("shape of e: ", e.shape)

Shape of a:  torch.Size([1, 6])
Shape of b:  torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d:  torch.Size([1, 1, 64])
shape of e:  torch.Size([1, 324])


In [66]:
learning_rate = 0.001
epochs = 20

In [67]:
model = SimpleRNN(len(vocab))

In [68]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [69]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 523.610247
Epoch: 2, Loss: 454.559391
Epoch: 3, Loss: 378.457108
Epoch: 4, Loss: 316.903921
Epoch: 5, Loss: 264.584342
Epoch: 6, Loss: 215.456879
Epoch: 7, Loss: 170.809186
Epoch: 8, Loss: 132.789889
Epoch: 9, Loss: 101.095114
Epoch: 10, Loss: 76.827672
Epoch: 11, Loss: 58.991031
Epoch: 12, Loss: 45.808988
Epoch: 13, Loss: 36.583895
Epoch: 14, Loss: 29.447548
Epoch: 15, Loss: 24.115584
Epoch: 16, Loss: 19.995111
Epoch: 17, Loss: 16.836260
Epoch: 18, Loss: 14.317162
Epoch: 19, Loss: 12.225659
Epoch: 20, Loss: 10.647782


In [70]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [76]:
predict(model, "What is the capital of france?")

paris


In [72]:
list(vocab.keys())

['<UNK>',
 'what',
 'is',
 'the',
 'capital',
 'of',
 'france',
 'paris',
 'germany',
 'berlin',
 'who',
 'wrote',
 'to',
 'kill',
 'a',
 'mockingbird',
 'harper-lee',
 'largest',
 'planet',
 'in',
 'our',
 'solar',
 'system',
 'jupiter',
 'boiling',
 'point',
 'water',
 'celsius',
 '100',
 'painted',
 'mona',
 'lisa',
 'leonardo-da-vinci',
 'square',
 'root',
 '64',
 '8',
 'chemical',
 'symbol',
 'for',
 'gold',
 'au',
 'which',
 'year',
 'did',
 'world',
 'war',
 'ii',
 'end',
 '1945',
 'longest',
 'river',
 'nile',
 'japan',
 'tokyo',
 'developed',
 'theory',
 'relativity',
 'albert-einstein',
 'freezing',
 'fahrenheit',
 '32',
 'known',
 'as',
 'red',
 'mars',
 'author',
 '1984',
 'george-orwell',
 'currency',
 'united',
 'kingdom',
 'pound',
 'india',
 'delhi',
 'discovered',
 'gravity',
 'newton',
 'how',
 'many',
 'continents',
 'are',
 'there',
 'on',
 'earth',
 '7',
 'gas',
 'do',
 'plants',
 'use',
 'photosynthesis',
 'co2',
 'smallest',
 'prime',
 'number',
 '2',
 'invented'